# Name That LLM – Ensemble: LinearSVC + ModernBERT (Stufe 2) + Sinkhorn

Inputs: Competition `name-that-llm`, Dataset `crtproduction/name-that-llm-open-pool`, Notebook-Output `linuszarse/name-that-llm-modernbert-stage2` (Ordner `model_stage2`). Kein Internet.

In [ ]:
W_SVM = 0.4          # Anteil SVM in der Mischung
W_LORA = 0.0        # Anteil Qwen-LoRA (0 = nicht verwendet); Rest: ModernBERT
SVM_T = 5.0          # Temperatur: softmax(SVM-Score * SVM_T)
SINKHORN = 0.0    # 0 = aus, 1 = voller Klassenausgleich
REAL_WEIGHT = 5
C = 0.1
SEED = 42

In [ ]:
import glob, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from scipy.special import softmax
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

T0 = time.time()
def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} nicht gefunden – Input eingebunden?"
    return Path(hits[0])

DATA_DIR = find("name-that-llm/train.csv").parent
POOL_PATH = find("ntl_open_pool.parquet")
BERT_DIR = find("model_stage2/config.json").parent
OUT_DIR = Path("/kaggle/working")
print(DATA_DIR, POOL_PATH, BERT_DIR, "| GPU:", torch.cuda.is_available())

## Features & Modell (aus `src/features.py` und `src/linear_large.py`)

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler

# Zeichen, deren Häufigkeit (pro 100 Wörter) als Feature dient
CHAR_COUNTS = {
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "curly_apos": "’",
    "curly_dquote": "“",
    "straight_apos": "'",
    "straight_dquote": '"',
    "colon": ":",
    "semicolon": ";",
    "comma": ",",
    "paren": "(",
    "question": "?",
    "exclam": "!",
    "ellipsis": "…",
}

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"\b\w+\b")


def stylometric_features(texts: pd.Series) -> pd.DataFrame:
    """Längen-, Satz-, Wort- und Interpunktionsstatistiken je Text."""
    rows = []
    for text in texts:
        words = _WORD.findall(text)
        n_words = max(len(words), 1)
        sents = [s for s in _SENT_SPLIT.split(text.strip()) if s]
        sent_lens = [len(_WORD.findall(s)) for s in sents] or [0]
        row = {
            "n_chars": len(text),
            "n_words": len(words),
            "n_sents": len(sents),
            "n_paras": text.count("\n\n") + 1,
            "words_per_sent": np.mean(sent_lens),
            "sent_len_std": np.std(sent_lens),
            "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
            "type_token_ratio": len({w.lower() for w in words}) / n_words,
            "upper_ratio": sum(c.isupper() for c in text) / max(len(text), 1),
            "non_ascii_ratio": sum(ord(c) > 127 for c in text) / max(len(text), 1),
            "digit_ratio": sum(c.isdigit() for c in text) / max(len(text), 1),
            "ends_with_period": float(text.rstrip().endswith(".")),
        }
        for name, ch in CHAR_COUNTS.items():
            row[f"per100w_{name}"] = 100 * text.count(ch) / n_words
        rows.append(row)
    return pd.DataFrame(rows, index=texts.index)


class StyloTransformer(BaseEstimator, TransformerMixin):
    """sklearn-Wrapper um stylometric_features (für Pipelines)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return stylometric_features(pd.Series(X)).to_numpy(dtype=float)


def word_tfidf(**kw) -> TfidfVectorizer:
    params = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def char_tfidf(**kw) -> TfidfVectorizer:
    params = dict(analyzer="char", ngram_range=(1, 5), min_df=3, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def text_features(use_stylo: bool = True):
    """FeatureUnion aus Wort-TF-IDF, Char-TF-IDF und (optional) skalierten Stilo-Features."""
    parts = [word_tfidf(), char_tfidf()]
    if use_stylo:
        parts.append(make_pipeline(StyloTransformer(), StandardScaler()))
    return make_union(*parts)

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.special import softmax
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


STYLO_WEIGHT = 0.1
SVM_TEMPERATURE = 5.0  # Skalierung der SVM-Scores für Pseudo-Wahrscheinlichkeiten (softmax)


class LargeVectorizer:
    """Wort- + Char-TF-IDF (begrenztes Vokabular, float32) + gewichtete Stilo-Features."""

    def __init__(self, max_word: int = 300_000, max_char: int = 500_000,
                 stylo_weight: float = STYLO_WEIGHT):
        self.word = word_tfidf(min_df=3, max_features=max_word, dtype=np.float32)
        self.char = char_tfidf(min_df=5, max_features=max_char, dtype=np.float32)
        self.scaler = StandardScaler()
        self.stylo_weight = stylo_weight

    def _stack(self, texts, fit: bool):
        if fit:
            w, c = self.word.fit_transform(texts), self.char.fit_transform(texts)
            s = self.scaler.fit_transform(stylometric_features(texts))
        else:
            w, c = self.word.transform(texts), self.char.transform(texts)
            s = self.scaler.transform(stylometric_features(texts))
        parts = [w, c]
        if self.stylo_weight:
            parts.append(sp.csr_matrix(s * self.stylo_weight))
        return sp.hstack(parts, format="csr", dtype=np.float32)

    def fit_transform(self, texts):
        return self._stack(texts, fit=True)

    def transform(self, texts):
        return self._stack(texts, fit=False)


def make_svm(C: float = 0.1):
    return LinearSVC(C=C, class_weight="balanced")


def svm_proba(clf, X, temperature: float = SVM_TEMPERATURE) -> np.ndarray:
    """Softmax über die SVM-Scores (für Ensembles; Temperatur später auf OOF kalibrieren)."""
    return softmax(clf.decision_function(X) * temperature, axis=1)


def sample_weights(n_pool: int, n_real: int, real_weight: float) -> np.ndarray:
    """Gewicht je Zeile: Pool = 1, echte train.csv-Texte = real_weight."""
    return np.concatenate([np.ones(n_pool), np.full(n_real, real_weight)]).astype(np.float32)

## Daten

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
pool = pd.read_parquet(POOL_PATH, columns=["text", "model"])
classes = np.sort(train["model"].unique())
assert set(pool["model"]) == set(classes)
print("train", train.shape, "| pool", pool.shape, "| test", test.shape)

## 1. LinearSVC (Pool + train.csv)

In [ ]:
vec = LargeVectorizer()
X_pool = vec.fit_transform(pool["text"])
X_train = vec.transform(train["text"])
X = sp.vstack([X_pool, X_train], format="csr")
y = np.concatenate([pool["model"].values, train["model"].values])
svm = make_svm(C).fit(X, y, sample_weight=sample_weights(len(pool), len(train), REAL_WEIGHT))
del X, X_pool
dec = svm.decision_function(vec.transform(test["text"]))
P_svm = pd.DataFrame(softmax(dec * SVM_T, axis=1), columns=svm.classes_)[classes].values
print(f"SVM fertig nach {(time.time() - T0) / 60:.1f} min")

## 2. ModernBERT Stufe 2

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(BERT_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).float().to(dev).eval()
bert_classes = [bert.config.id2label[i] for i in range(bert.config.num_labels)]
assert set(bert_classes) == set(classes)

enc = tok(list(test["text"]), truncation=True, max_length=384)
order = np.argsort([len(x) for x in enc["input_ids"]])
items = [{k: enc[k][i] for k in enc} for i in order]
dl = DataLoader(items, batch_size=64, collate_fn=DataCollatorWithPadding(tok, pad_to_multiple_of=8))
out = []
with torch.no_grad():
    for b in dl:
        b = {k: v.to(dev) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            out.append(torch.softmax(bert(**b).logits.float(), -1).cpu().numpy())
P_bert = np.empty((len(test), len(bert_classes)), dtype=np.float32)
P_bert[order] = np.concatenate(out)
P_bert = pd.DataFrame(P_bert, columns=bert_classes)[classes].values
print(f"BERT fertig nach {(time.time() - T0) / 60:.1f} min")

## Mischung + Sinkhorn-Ausgleich

In [ ]:
P = W_SVM * P_svm + (1 - W_SVM - W_LORA) * P_bert
if W_LORA:
    P = P + W_LORA * P_lora

def sinkhorn(P, strength, iters=100):
    Q = P.copy(); target = np.full(P.shape[1], len(P) / P.shape[1])
    for _ in range(iters):
        Q *= target / Q.sum(0); Q /= Q.sum(1, keepdims=True)
    return np.exp((1 - strength) * np.log(P + 1e-12) + strength * np.log(Q + 1e-12))

if SINKHORN and len(test) >= 50:      # nicht auf den 3 Beispieltexten
    P = sinkhorn(P, SINKHORN)
pred = classes[P.argmax(1)]
print("Übereinstimmung SVM/BERT:", round(float((P_svm.argmax(1) == P_bert.argmax(1)).mean()), 3))

## Submission

In [ ]:
sub = pd.DataFrame({"id": test["id"], "model": pred})
assert list(sub.columns) == ["id", "model"]
assert sub.notna().all().all(), "Nullwerte"
assert sub["id"].is_unique, "doppelte IDs"
assert len(sub) == len(test) and set(sub["id"]) == set(test["id"]), "nicht jede Test-ID genau einmal"
assert set(sub["model"]) <= set(classes) and len(classes) == 18, "unbekanntes Label"
sub.to_parquet(OUT_DIR / "submission.parquet", index=False)
print(sub["model"].value_counts().head(10), f"\ngesamt {(time.time() - T0) / 60:.1f} min")
sub.head()